# Colab Training Guide

This notebook shows how to run the Phase 1 cached training pipeline using the prepared `code.zip` and `data_bundle.zip` bundles. The dataset is already preprocessed and cached, so training runs without DICOM decoding in Colab.

## Step 1: Mount Google Drive and access bundles

Put `code.zip` and `data_bundle.zip` in your Drive folder named `project`.
Then run the cells below to mount Drive and unzip the bundles into the Colab VM.

In [ ]:
import os
import pathlib

# Adjust these paths if you uploaded the files to a different location.
CODE_ZIP = '/content/code.zip'
DATA_ZIP = '/content/data_bundle.zip'
DEST_DIR = '/content/project'

os.makedirs(DEST_DIR, exist_ok=True)
!unzip -q $CODE_ZIP -d $DEST_DIR
!unzip -q $DATA_ZIP -d $DEST_DIR

print('Unzipped code and data bundles to', DEST_DIR)
print('Contents:')
!find $DEST_DIR -maxdepth 2 -type f | sort | sed -n '1,50p'

## Step 2: Install dependencies

Install Python dependencies required by the pipeline. If your Colab runtime already has these packages, this step may be fast.

In [ ]:
!python3 -m pip install --quiet torch torchvision torchaudio pandas numpy opencv-python pydicom tqdm

## Step 3: Configure the environment

Set the original data root used to generate cache keys. This ensures the cached `.npy` files can be found even though the DICOM files are not present in Colab.

In [ ]:
import os
cache_origin_path = '/content/project/data/cache_origin.txt'
with open(cache_origin_path, 'r') as f:
    origin = f.read().strip()
os.environ['NLBS_ORIGINAL_DATA_ROOT'] = origin
print('NLBS_ORIGINAL_DATA_ROOT =', origin)

## Step 4: Run the cached training smoke test

This script uses cached `.npy` images only, without reading DICOM files. It will verify the data pipeline and run a small training loop.

In [ ]:
import sys
sys.path.append('/content/project/Phase1_DL')
!PYTHONPATH=/content/project/Phase1_DL python3 /content/project/scripts/test_cache_and_train.py

## Step 5: Train on the balanced cohort

If you want to train on the balanced abnormal/normal cohort, use `patient_manifest_balanced.csv` and the cached data already included in the bundle. Update the training script or dataset target accordingly.

## Step 5: Run Full Training with Live Progress

This runs a full training loop on the balanced cohort with live epoch and batch progress.

In [ ]:
!PYTHONPATH=/content/project/Phase1_DL python3 /content/project/scripts/train_colab.py --epochs 10 --batch-size 8 --manifest balanced
